In [1]:
# ============================================================
# AML23703 - QUANTUM COMPUTING
# TUTORIAL 4: MULTI-QUBIT GATES AND CIRCUIT COMPOSITION
#
# EASY   : CNOT gate with control qubit |1>
# MEDIUM : Bell state with 1024 shots
# HARD   : 3-qubit GHZ state
# REAL-WORLD : Classical Full Adder using Toffoli and CNOT
# ============================================================


# ------------------------------------------------------------
# INSTALL REQUIRED LIBRARIES
# ------------------------------------------------------------

!pip install qiskit qiskit-aer -q


# ------------------------------------------------------------
# IMPORT LIBRARIES
# ------------------------------------------------------------

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator


# Create simulator
simulator = AerSimulator()


# ============================================================
# EASY
# Implement CNOT with control qubit |1>
# and verify that target qubit flips
# ============================================================

print("\n" + "=" * 60)
print("EASY: CNOT GATE")
print("=" * 60)

# 2 qubits and 2 classical bits
qc_easy = QuantumCircuit(2, 2)

# Set control qubit q0 to |1>
qc_easy.x(0)

# CNOT
# q0 = control
# q1 = target
qc_easy.cx(0, 1)

# Measure both qubits
qc_easy.measure([0, 1], [0, 1])

# Run circuit
result_easy = simulator.run(
    qc_easy,
    shots=1
).result()

counts_easy = result_easy.get_counts()

print("\nCNOT Circuit:")
print(qc_easy)

print("Measurement result:")
print(counts_easy)

print("\nExpected result: 11")
print("Explanation: Control qubit is 1, so target qubit flips from 0 to 1.")


# ============================================================
# MEDIUM
# Bell State
# Run 1024 shots
# Verify outcomes are only 00 and 11
# ============================================================

print("\n" + "=" * 60)
print("MEDIUM: BELL STATE")
print("=" * 60)

# Create 2-qubit Bell circuit
qc_medium = QuantumCircuit(2, 2)

# Put first qubit into superposition
qc_medium.h(0)

# Entangle second qubit with first
qc_medium.cx(0, 1)

# Measure both qubits
qc_medium.measure([0, 1], [0, 1])

# Run 1024 shots
result_medium = simulator.run(
    qc_medium,
    shots=1024
).result()

counts_medium = result_medium.get_counts()

print("\nBell State Circuit:")
print(qc_medium)

print("Measurement results:")
print(counts_medium)

print("\nExpected:")
print("Only 00 and 11 should appear.")

print("\nExplanation:")
print("The two qubits are entangled, so they give the same result.")
print("Approximately 50% should be 00 and 50% should be 11.")


# ============================================================
# HARD
# 3-Qubit GHZ State
# Confirm three-way correlation
# ============================================================

print("\n" + "=" * 60)
print("HARD: 3-QUBIT GHZ STATE")
print("=" * 60)

# Create 3-qubit GHZ circuit
qc_hard = QuantumCircuit(3, 3)

# Create superposition on first qubit
qc_hard.h(0)

# Entangle q0 with q1
qc_hard.cx(0, 1)

# Entangle q0 with q2
qc_hard.cx(0, 2)

# Measure all three qubits
qc_hard.measure([0, 1, 2], [0, 1, 2])

# Run 1024 shots
result_hard = simulator.run(
    qc_hard,
    shots=1024
).result()

counts_hard = result_hard.get_counts()

print("\nGHZ Circuit:")
print(qc_hard)

print("Measurement results:")
print(counts_hard)

print("\nExpected:")
print("Only 000 and 111 should appear.")

print("\nExplanation:")
print("All three qubits are entangled.")
print("If one qubit is 0, the others are also 0.")
print("If one qubit is 1, the others are also 1.")


# ============================================================
# REAL-WORLD
# Classical Full Adder using Toffoli and CNOT gates
#
# Inputs:
# A, B, Cin
#
# Outputs:
# Sum = A XOR B XOR Cin
# Carry = AB + Cin(A XOR B)
# ============================================================

print("\n" + "=" * 60)
print("REAL-WORLD: CLASSICAL FULL ADDER")
print("=" * 60)


# ------------------------------------------------------------
# Function to implement full adder
# ------------------------------------------------------------

def full_adder(a, b, cin):

    # 5 qubits:
    #
    # q0 = A
    # q1 = B
    # q2 = Carry-in
    # q3 = Sum
    # q4 = Carry

    qc = QuantumCircuit(5, 2)

    # Set input A
    if a == 1:
        qc.x(0)

    # Set input B
    if b == 1:
        qc.x(1)

    # Set Carry-in
    if cin == 1:
        qc.x(2)


    # --------------------------------------------------------
    # SUM
    #
    # Sum = A XOR B XOR Cin
    # --------------------------------------------------------

    qc.cx(0, 3)
    qc.cx(1, 3)
    qc.cx(2, 3)


    # --------------------------------------------------------
    # CARRY
    #
    # Carry = AB + Cin(A XOR B)
    #
    # Toffoli q0,q1 -> q4
    # Toffoli q2,q3 -> q4
    #
    # q3 currently contains A XOR B XOR Cin,
    # so for a classical truth-table demonstration
    # we use an additional temporary qubit for A XOR B.
    # --------------------------------------------------------

    # Reset q3 and create A XOR B temporarily
    qc.reset(3)

    # q3 = A XOR B
    qc.cx(0, 3)
    qc.cx(1, 3)

    # AB contribution
    qc.ccx(0, 1, 4)

    # Cin(A XOR B) contribution
    qc.ccx(2, 3, 4)

    # Recreate final sum
    qc.cx(0, 3)
    qc.cx(1, 3)
    qc.cx(2, 3)


    # Measure:
    # q3 -> Sum
    # q4 -> Carry
    qc.measure(3, 0)
    qc.measure(4, 1)

    return qc


# ------------------------------------------------------------
# Run all 8 possible input combinations
# ------------------------------------------------------------

print("\nFULL ADDER TRUTH TABLE")
print("-" * 60)
print(" A   B   Cin   |   Sum   Carry")
print("-" * 60)


for a in [0, 1]:
    for b in [0, 1]:
        for cin in [0, 1]:

            circuit = full_adder(a, b, cin)

            result = simulator.run(
                circuit,
                shots=1
            ).result()

            counts = result.get_counts()

            # Get output
            output = list(counts.keys())[0]

            # Because classical bit 1 is carry
            # and classical bit 0 is sum,
            # output is written as CarrySum
            carry = output[0]
            sum_bit = output[1]

            print(
                f" {a}   {b}    {cin}    |    "
                f"{sum_bit}      {carry}"
            )


# ------------------------------------------------------------
# Display circuit for one example
# A=1, B=1, Cin=1
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FULL ADDER CIRCUIT FOR A=1, B=1, Cin=1")
print("=" * 60)

example = full_adder(1, 1, 1)

print(example)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("TUTORIAL 4 COMPLETED")
print("=" * 60)

print("""
EASY:
CNOT flips the target qubit when the control is |1>.

MEDIUM:
Bell state produces only 00 and 11 because the qubits
are entangled.

HARD:
GHZ state produces only 000 and 111, showing
three-qubit correlation.

REAL-WORLD:
A classical full adder was implemented using
CNOT and Toffoli gates.
""")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.3 MB/s eta 0:00:00

EASY: CNOT GATE

CNOT Circuit:
     ┌───┐     ┌─┐   
q_0: ┤ X ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 
Measurement result:
{'11': 1}

Expected result: 11
Explanation: Control qubit is 1, so target qubit flips from 0 to 1.

MEDIUM: BELL STATE

Bell State Circuit:
     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 
Measurement results:
{'11': 509, '00': 515}

Expected:
Only 00 and 11 should appear.

Explanation:
The two qubits are entangled, so they give the same result.
Approximately 50% should be 00